In [1]:
import re
import unicodedata

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from sklearn.metrics import accuracy_score, classification_report, f1_score
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [2]:
def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


model_name = "cross-encoder/quora-distilroberta-base"
model = CrossEncoder(model_name, device=str(device))
id2label = {0: "not_duplicate", 1: "duplicate"}

print("model:", model_name)
print("num_labels:", getattr(model.config, "num_labels", None))


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/quora-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model: cross-encoder/quora-distilroberta-base
num_labels: 1


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

norm_sentence1 = [normalize_text(x) for x in sent1]
norm_sentence2 = [normalize_text(x) for x in sent2]
normalized_pairs = list(zip(norm_sentence1, norm_sentence2))

changed_s1 = sum(a != b for a, b in zip(sent1, norm_sentence1))
changed_s2 = sum(a != b for a, b in zip(sent2, norm_sentence2))

print("num_examples:", len(y_true))
print("positive_rate:", float(y_true.mean()))
print("sentence1_changed_by_normalization:", changed_s1)
print("sentence2_changed_by_normalization:", changed_s2)


num_examples: 408
positive_rate: 0.6838235294117647
sentence1_changed_by_normalization: 0
sentence2_changed_by_normalization: 1


In [5]:
batch_size = 64
raw_scores = model.predict(
    normalized_pairs,
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True,
)
raw_scores = np.asarray(raw_scores)

if raw_scores.ndim == 2:
    y_pred = np.argmax(raw_scores, axis=1).astype(int)
elif raw_scores.ndim == 1:
    y_pred = (raw_scores >= 0.5).astype(int)
else:
    raise ValueError(f"Unexpected score shape: {raw_scores.shape}")

print("raw_scores_shape:", raw_scores.shape)
print("done")


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

raw_scores_shape: (408,)
done


In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.6838235294117647, 'f1': 0.7667269439421338}
                precision    recall  f1-score   support

not_paraphrase       0.50      0.52      0.51       129
    paraphrase       0.77      0.76      0.77       279

      accuracy                           0.68       408
     macro avg       0.64      0.64      0.64       408
  weighted avg       0.69      0.68      0.69       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("original_sentence1:", sent1[i])
    print("normalized_sentence1:", norm_sentence1[i])
    print("original_sentence2:", sent2[i])
    print("normalized_sentence2:", norm_sentence2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", id2label[int(y_pred[i])])
    if raw_scores.ndim == 2:
        print("scores:", raw_scores[i].tolist())
    else:
        print("score:", float(raw_scores[i]))


original_sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
normalized_sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
original_sentence2: " The foodservice pie business does not fit our long-term growth strategy .
normalized_sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 label: duplicate
score: 0.9779019355773926
original_sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
normalized_sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
original_sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
normalized_sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using hi

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("original_sentence1:", sent1[i])
    print("normalized_sentence1:", norm_sentence1[i])
    print("original_sentence2:", sent2[i])
    print("normalized_sentence2:", norm_sentence2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    if raw_scores.ndim == 2:
        print("scores:", raw_scores[i].tolist())
    else:
        print("score:", float(raw_scores[i]))


num_errors: 129
idx: 5
original_sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
normalized_sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
original_sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
normalized_sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
true: 1 pred: 0
score: 0.015424183569848537
idx: 6
original_sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
normalized_sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
original_sentence2: The Institute said dioxin levels in the environment have fallen by as much as 

In [9]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "preprocessing": "nfkc_whitespace_normalization",
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'cross-encoder/quora-distilroberta-base',
 'device': 'mps',
 'preprocessing': 'nfkc_whitespace_normalization',
 'num_examples': 408,
 'accuracy': 0.6838235294117647,
 'f1': 0.7667269439421338}